# Нормализация числовых записей

`normalize_numbers(text)` принимает всю строку и возвращает строку с нормализованными числовыми фрагментами. Используется только стандартная библиотека Python.

Поддерживает английские числительные, словесные десятичные числа, дроби, смешанные числа, LaTeX, Unicode, проценты и известные опечатки. Выражения не вычисляются, дроби не сокращаются.

Правило для запятой: одиночная запятая между цифрами считается десятичной (`6,079 → 6.079`, в том числе неоднозначное `1,440 → 1.440`). Цепочки вроде `6,10,14,18` сохраняются как списки. Неизвестные опечатки и отношения через `:` сохраняются.

Вход должен быть строкой. Для столбцов с пропусками сначала обработайте их, например: `df["StudentExplanation"].fillna("").apply(normalize_numbers)`.


In [22]:
import re
import unicodedata
import pandas as pd
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


# Словарь: английские названия чисел от zero до nineteen -> целые числа от 0 до 19.
_NN_SMALL = dict(zip(
    'zero one two three four five six seven eight nine ten eleven twelve '
    'thirteen fourteen fifteen sixteen seventeen eighteen nineteen'.split(),
    range(20),
))
# Дополнительные ключи словаря: nought, naught и nil тоже обозначают число 0.
_NN_SMALL.update(nought=0, naught=0, nil=0)
# Словарь: названия десятков twenty, thirty, ..., ninety -> числа 20, 30, ..., 90.
_NN_TENS = dict(zip(
    'twenty thirty forty fifty sixty seventy eighty ninety'.split(),
    range(20, 100, 10),
))
# Словарь: названия крупных разрядов -> множители (thousand -> 1000, million -> 1000000).
_NN_SCALES = {'thousand': 1000, 'million': 10**6, 'billion': 10**9}
# Словарь: порядковые числительные first, second, ..., twentieth -> их номера от 1 до 20.
# Этот справочник используется для названий знаменателей; отдельно порядковые слова не преобразуются.
_NN_ORDINALS = dict(zip(
    'first second third fourth fifth sixth seventh eighth ninth tenth eleventh '
    'twelfth thirteenth fourteenth fifteenth sixteenth seventeenth eighteenth '
    'nineteenth twentieth'.split(), range(1, 21),
))
# Дополняем словарь порядковыми словами для 30, 40, ..., 90, 100 и 1000.
_NN_ORDINALS.update(thirtieth=30, fortieth=40, fiftieth=50, sixtieth=60,
                    seventieth=70, eightieth=80, ninetieth=90,
                    hundredth=100, thousandth=1000)
# Словарь: название доли -> знаменатель дроби (third -> 3, fifth -> 5); сначала берём значения от 3.
_NN_DENOMS = {word: n for word, n in _NN_ORDINALS.items() if n >= 3}
# Добавляем множественное число названий долей: thirds -> 3, fifths -> 5 и т. д.
_NN_DENOMS.update({word + 's': n for word, n in list(_NN_DENOMS.items())})
# Добавляем особые названия: half/halves -> знаменатель 2, quarter/quarters -> знаменатель 4.
_NN_DENOMS.update(half=2, halves=2, quarter=4, quarters=4)
# Словарь: готовый символ дроби Unicode -> строка с дробью (например, ⅓ -> "1/3", ½ -> "1/2").
_NN_FRACTIONS = dict(zip(
    '¼½¾⅐⅑⅒⅓⅔⅕⅖⅗⅘⅙⅚⅛⅜⅝⅞',
    ['1/4', '1/2', '3/4', '1/7', '1/9', '1/10', '1/3', '2/3',
     '1/5', '2/5', '3/5', '4/5', '1/6', '5/6', '1/8', '3/8', '5/8', '7/8'],
))

# Словарь: вариант слова с опечаткой -> правильное написание (threee -> three); заполняется циклом ниже.
_NN_TYPOS = {}
# В словаре ниже ключ — правильное слово, значение — строка вариантов его написания через пробел.
for _nn_correct, _nn_variants in {
    'one': '0ne onee ome onr oane', 'two': 'twoo twi rwo',
    'three': 'threee threeee thrre thre hree threr tree', 'four': 'fourr fiur',
    'five': 'fuve fiv firve fivee', 'six': 'sixx dix sux sixs',
    'eight': 'eightt eght', 'nine': 'ninee ninr nime', 'ten': 'tem tenn',
    'fifteen': 'fiften fifteenn fifthteen', 'twenty': 'tweny',
    'thirty': 'therty', 'hundred': 'hundredd',
    'half': 'halff hakf hslf halfs haalf alf hasf hlf',
    'quarter': 'qaurter quater', 'quarters': 'quaters',
    'third': 'thirdd thired thrid thirs thitd tird therd',
    'thirds': 'thirdss tihrds therds', 'fifth': 'fith fifthh',
    'fifths': 'fiths fifthss', 'eighth': 'eighthh',
    'eighths': 'eighthss eigths eghts', 'ninth': 'nineth',
    'ninths': 'nineths ninthes ninthss niths nithes ninthd',
    'tenth': 'thenth', 'tenths': 'thenths tenthes enths',
    'twelfth': 'twelth twelthe twelvth twelvith', 'twelfths': 'twelths',
    'fifteenths': 'fifteeths fiteenths fifthteenths fifthteens fifthtenths fiftinfs',
    'thirtieth': 'thirtyth', 'thirtieths': 'thirtyths',
    'hundredth': 'hundreth', 'hundredths': 'hundreths hundereths',
    'thousandths': 'thousanths thousadnths',
}.items():
    _NN_TYPOS.update(dict.fromkeys(_nn_variants.split(), _nn_correct))


def _nn_alternatives(words):
    return '(?:' + '|'.join(
        re.escape(w) 
        for w in sorted(words, key=len, reverse=True)) + ')'


# Строка-шаблон regex: варианты названий единиц от one до nine, соединённые через «или».
_NN_UNIT = _nn_alternatives(list(_NN_SMALL)[1:10])
# Строка-шаблон regex: числа меньше 100, например five, nineteen, twenty four, twenty-four.
_NN_UNDER100 = (
    rf'(?:{_nn_alternatives(_NN_TENS)}(?:(?: +|-){_NN_UNIT})?'
    rf'|{_nn_alternatives(_NN_SMALL)})'
)
# Строка-шаблон regex: числа со словом hundred, например one hundred and five, а также числа меньше 100.
_NN_UNDER1000 = (
    rf'(?:(?:{_NN_UNDER100}(?: +|-))?hundred'
    rf'(?:(?: +|-)(?:and(?: +|-))?{_NN_UNDER100})?|{_NN_UNDER100})'
)
# Строка-шаблон regex: составные целые числительные, включая thousand, million и billion.
_NN_CARDINAL = (
    rf'{_NN_UNDER1000}(?:(?: +|-){_nn_alternatives(_NN_SCALES)}'
    rf'(?:(?: +|-)(?:and(?: +|-))?{_NN_UNDER1000})?)*'
)
# Скомпилированный шаблон: ищет словесные целые числа, проверяя отсутствие соседних латинских букв.
_NN_WORD_NUMBER = re.compile(rf'(?<![A-Za-z]){_NN_CARDINAL}(?![A-Za-z])', re.I)
# Скомпилированный шаблон: десятичные числа со словом point; группы whole и fraction хранят целую и дробную части.
_NN_DECIMAL_WORDS = re.compile(
    rf'(?<![A-Za-z0-9])(?:(?P<whole>{_NN_CARDINAL}|\d+)[ ]+)?point[ ]+'
    rf'(?P<fraction>(?:{_NN_UNDER100}|\d+)'
    rf'(?:(?: +|-)(?:{_NN_UNDER100}|\d+))*)(?![A-Za-z])', re.I,
)
# Скомпилированный шаблон: находит числовые слова или цифры внутри части после point (zero seven nine -> 0, 7, 9).
_NN_DECIMAL_PART = re.compile(rf'{_NN_UNDER100}|\d+', re.I)
# Строка-шаблон regex: все названия знаменателей из _NN_DENOMS, например third, thirds, half.
_NN_DENOM_PATTERN = _nn_alternatives(_NN_DENOMS)
# Скомпилированный шаблон: ищет варианты написания, перечисленные среди ключей словаря _NN_TYPOS.
_NN_TYPO_PATTERN = re.compile(
    rf'(?<![A-Za-z]){_nn_alternatives(_NN_TYPOS)}(?![A-Za-z])', re.I,
)


def _nn_cardinal_value(words):
    total = current = 0
    for word in re.split(r'[ -]+', words.lower()):
        if word in _NN_SMALL:
            current += _NN_SMALL[word]
        elif word in _NN_TENS:
            current += _NN_TENS[word]
        elif word == 'hundred':
            current = (current or 1) * 100
        elif word in _NN_SCALES:
            total += (current or 1) * _NN_SCALES[word]
            current = 0
    return total + current


def _nn_decimal_value(match):
    whole = match['whole'] or '0'
    whole = str(int(whole)) if whole.isdecimal() else str(_nn_cardinal_value(whole))
    # После point слова zero seven nine означают последовательность цифр 079.
    digits = ''.join(
        part if part.isdecimal() else str(_nn_cardinal_value(part))
        for part in _NN_DECIMAL_PART.findall(match['fraction'])
    )
    return whole + '.' + digits


def normalize_numbers(text: str) -> str:
    """Принимает целую строку и возвращает строку с нормализованными числами.

    Английские числительные, дроби, mixed numbers, проценты и известные
    опечатки приводятся к цифрам. Выражения не вычисляются, дроби не
    сокращаются, конечные нули десятичной части сохраняются.

    Одиночная запятая между цифрами считается десятичной: 6,079 -> 6.079
    (в том числе неоднозначное 1,440 -> 1.440). Списки 6,10,14 сохраняются.
    :, неизвестные опечатки и надстрочные степени сохраняются без догадок.
    Не предназначена для произвольных дат, программного кода или URL.
    """
    if not isinstance(text, str):
        raise TypeError('normalize_numbers expects a string')

    # Сначала многострочные дроби, затем общая очистка пробелов.
    text = re.sub(r'(\d+)[ \t]*\n[ \t]*[-_]{2,}[ \t]*\n[ \t]*(\d+)', r'\1/\2', text)
    text = re.sub(r'\s+', ' ', text.replace('\u200b', '')).strip()
    text = ''.join(str(unicodedata.decimal(c)) if c.isdecimal() else c for c in text)
    text = _NN_TYPO_PATTERN.sub(lambda m: _NN_TYPOS[m[0].lower()], text)

    # LaTeX и готовые символы дробей. Смешанное 3⅓ требует пробела.
    text = re.sub(
        r'\\(?:dfrac|tfrac|frac)\s*\{\s*([+-]?\d+)\s*\}\s*\{\s*([+-]?\d+)\s*\}',
        r'\1/\2', text,
    )
    text = re.sub(r'\\[()\[\]]', '', text)
    text = re.sub(r'\\(?:times|cdot)\b', '*', text)
    text = re.sub(r'\\div\b', '/', text)
    text = re.sub(rf'(?<=\d)(?=[{"".join(_NN_FRACTIONS)}])', ' ', text)
    text = ''.join(_NN_FRACTIONS.get(c, c) for c in text)
    # Таблица замены надстрочных цифр ⁰¹²³⁴⁵⁶⁷⁸⁹ на обычные 0123456789; ниже применяется только перед дробной чертой.
    superscripts = str.maketrans('⁰¹²³⁴⁵⁶⁷⁸⁹', '0123456789')
    text = re.sub(r'[⁰¹²³⁴⁵⁶⁷⁸⁹]+(?=\s*/)', lambda m: m[0].translate(superscripts), text)


    text = _NN_DECIMAL_WORDS.sub(_nn_decimal_value, text)
    text = _NN_WORD_NUMBER.sub(lambda m: str(_nn_cardinal_value(m[0])), text)
    text = re.sub(r'(?<!\w)o\.(?=\d)', '0.', text, flags=re.I)

    # x между операндами раскрываем до разбора дробей: 2x5 over 3.
    text = re.sub(r'[×✖]️?', '*', text)
    text = re.sub(r'[÷➗]️?', '/', text)
    text = re.sub(r'(?<=\d)\s*[xX]\s*(?=\d)', '*', text)
    text = re.sub(r'(?<=\d)((?:rds?|ths?|s)(?:[’\x27]s)?)\s*[xX]\s*(?=\d)',
                  r'\1*', text, flags=re.I)
    text = re.sub(rf'\b({_NN_DENOM_PATTERN})\s*[xX]\s*(?=\d)',
                  r'\1*', text, flags=re.I)

    
    text = re.sub(
        rf'(?:/|\bover\b)\s*(\d+|an?)\s+({_NN_DENOM_PATTERN})(?![A-Za-z])',
        lambda m: '/ (' + (m[1] if m[1].isdigit() else '1') + '/' +
                  str(_NN_DENOMS[m[2].lower()]) + ')', text, flags=re.I,
    )
    text = re.sub(r'/\s*(half|quarter)\b',
                  lambda m: '/ (1/' + str(_NN_DENOMS[m[1].lower()]) + ')',
                  text, flags=re.I)
    text = re.sub(rf'\ban?\s+({_NN_DENOM_PATTERN})(?![A-Za-z])',
                  lambda m: '1/' + str(_NN_DENOMS[m[1].lower()]), text, flags=re.I)
    text = re.sub(
        rf'(?<![\w.])(?P<n>\d+)\s*(?:[/ -]\s*)?(?P<d>{_NN_DENOM_PATTERN})(?:[’\x27]s)?(?![A-Za-z])',
        lambda m: m['n'] + '/' + str(_NN_DENOMS[m['d'].lower()]), text, flags=re.I,
    )
    text = re.sub(r'\b(?:half|quarter)\b',
                  lambda m: '1/' + str(_NN_DENOMS[m[0].lower()]), text, flags=re.I)
    text = re.sub(r'(?<![\w/.])(\d+)\s+(\d+)(?:rds?|ths?)(?:[’\x27]s)?\b', r'\1/\2', text, flags=re.I)
    text = re.sub(r'(\d+\s*/\s*\d+)(?:rds?|ths?|s)(?:[’\x27]s)?\b', r'\1', text, flags=re.I)
    text = re.sub(
        r'(?<![\w/.])(\d+)\s*(?:over|out\s+of(?:\s+every)?)\s*(\d+)(?!\w)',
        r'\1/\2', text, flags=re.I,
    )

 
    text = re.sub(r'(?<=\d)\s*[/\\|;’\x27]\s*(?=\d)', '/', text)
    def mixed_number(match):
        starts_new_equation = (
            text[:match.start()].rstrip().endswith('=')
            and re.match(r'(?:\s*[-+*/]\s*\d+)*\s*=', text[match.end():])
        )
        return match[0] if starts_new_equation else match[1] + ' ' + match[2]

    text = re.sub(
        r'(?<![\w/.])(\d+)\s*(?:(?:wholes?\s+)?and|&)\s*(\d+/\d+)(?!\d)',
        mixed_number, text, flags=re.I,
    )
    text = re.sub(r'(?<![\w/.])(\d+),(\d+/\d+)(?!\d|\s*[=+*/-])', r'\1 \2', text)

    text = re.sub(r'(?<=\d)[•·](?=\d)', '.', text)
    def decimal_comma(match):
        raw = match[0]
        is_equation_boundary = (
            text[:match.start()].rstrip().endswith('=')
            and re.match(r'(?:\s*[-+*/]\s*\d+(?:\.\d+)?)*\s*=', text[match.end():])
        )
        return raw.replace(',', '.') if raw.count(',') == 1 and not is_equation_boundary else raw

    text = re.sub(
        r'(?<![\w.,])\d+(?:,\d+)+(?![\w,]|\.\d)',
        decimal_comma, text,
    )
    text = re.sub(r'(?<![\w.])\.(?=\d)', '0.', text)



    text = re.sub(r'\b(?:negative|minus)\s+(?=\d)', '-', text, flags=re.I)
    text = re.sub(r'\b(?:positive|plus)\s+(?=\d)', '+', text, flags=re.I)
    text = text.replace('−', '-')
    text = re.sub(r'[–—](?=\s*\d)', '-', text)
    text = re.sub(r'(?<=\d)\s*(?:percent|per\s+cent)\b', '%', text, flags=re.I)
    text = re.sub(r'(?<=\d)\s+%', '%', text)
    text = re.sub(r'(?<!\w)(\d+)\s*(?:remainder|rem|r)\s*(\d+)(?!\w)',
                  r'\1 remainder \2', text, flags=re.I)
    text = re.sub(r'(?<=\d)\s*/\s*(?=\d)', '/', text)

    text = re.sub(
        r'(?<![\w.])\d+(?:\.\d+)?(?!\w|\.\d)',
        lambda m: str(int(m[0].split('.')[0])) +
                  ('.' + m[0].split('.', 1)[1] if '.' in m[0] else ''), text,
    )
    return re.sub(r'\s+', ' ', text).strip()


In [23]:
examples = [
    "I think six point two is greater than 6,079 and 6•0001.",
    "one third of the cake is not shaded.",
    "3 ⅓ is three and one third.",
    "0ne third and threee nineths.",
]

for text in examples:
    print(text)
    print("->", normalize_numbers(text))
    print()






I think six point two is greater than 6,079 and 6•0001.
-> I think 6.2 is greater than 6.079 and 6.0001.

one third of the cake is not shaded.
-> 1/3 of the cake is not shaded.

3 ⅓ is three and one third.
-> 3 1/3 is 3 1/3.

0ne third and threee nineths.
-> 1/3 and 3/9.



In [24]:
# DataFrame test хранит все строки и столбцы, прочитанные из train.csv.
test = pd.read_csv('train.csv')

In [25]:
normalize_numbers(test['StudentExplanation'].iloc[1])

'1/3 because 6/9 is 2/3 and 1/3 is not shaded.'

In [26]:
print(test['StudentExplanation'].iloc[1])

1 / 3 because 6 over 9 is 2 thirds and 1 third is not shaded.


In [27]:
# Список ключей словаря с индексами 1–9: слова one, two, ..., nine.
list(_NN_SMALL)[1:10]

['one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']

In [28]:

def map_at_3(y_true, top3_predictions):
    """Один верный класс на строку: 1 за первое место, 1/2 за второе, 1/3 за третье."""
    y_true = np.asarray(y_true, dtype=object)
    predicted = np.asarray(top3_predictions, dtype=object)
    if y_true.ndim != 1 or len(y_true) == 0 or predicted.shape != (len(y_true), 3):
        raise ValueError('Нужны непустой y_true и предсказания формы (число строк, 3).')
    matches = predicted == y_true[:, None]
    # argmax находит первое совпадение; повтор класса не увеличивает оценку.
    reciprocal_rank = np.where(matches.any(axis=1), 1.0 / (matches.argmax(axis=1) + 1), 0.0)
    return float(reciprocal_rank.mean())




def make_baseline_split(y, groups, random_state=42, validation_size=0.2):
    """Одно общее разбиение: одинаковые исходные тексты остаются в одной части."""
    y, groups = np.asarray(y), np.asarray(groups)
    splitter = GroupShuffleSplit(n_splits=1, test_size=validation_size, random_state=random_state)
    train_idx, valid_idx = next(splitter.split(np.zeros(len(y)), y, groups))


    unseen = np.setdiff1d(y[valid_idx], y[train_idx])
    if len(unseen):
        groups_to_move = np.unique(groups[valid_idx[np.isin(y[valid_idx], unseen)]])
        move_mask = np.isin(groups[valid_idx], groups_to_move)
        train_idx = np.concatenate([train_idx, valid_idx[move_mask]])
        valid_idx = valid_idx[~move_mask]
    if len(valid_idx) == 0:
        raise ValueError('После переноса редких классов validation пуст: нужно больше данных.')
    return np.sort(train_idx), np.sort(valid_idx)


def make_text_baseline(random_state=42):
    """Свежая модель с одинаковыми параметрами для обоих вариантов текстов."""
    return Pipeline([
        ('tfidf', FeatureUnion([
            ('words', TfidfVectorizer(
                ngram_range=(1, 2), min_df=2, max_features=40_000,
                token_pattern=r'(?u)\b\w+\b', sublinear_tf=True,
            )),
            ('chars', TfidfVectorizer(
                analyzer='char', ngram_range=(3, 5), min_df=2,
                max_features=60_000, sublinear_tf=True,
            )),
        ])),
        ('classifier', LinearSVC(C=1.0, dual='auto', max_iter=10_000, random_state=random_state)),
    ])


def compare_normalization_baseline(normalize_fn, csv_path='train.csv', random_state=42):
    """Сравнить original/normalized на одном holdout без настройки по validation."""

    data = pd.read_csv(Path(csv_path), keep_default_na=False)
    required = {'StudentExplanation', 'Category', 'Misconception'}
    if not required.issubset(data.columns):
        raise ValueError(f'В CSV отсутствуют столбцы: {sorted(required - set(data.columns))}')

    original = data['StudentExplanation'].fillna('').astype(str).reset_index(drop=True)
    target = (data['Category'].astype(str) + ':' + data['Misconception'].astype(str)).to_numpy()
    if len(np.unique(target)) < 3:
        raise ValueError('Для сравнения MAP@3 нужны как минимум три класса в данных.')

    # Группируем дубликаты по исходному тексту, игнорируя регистр и лишние пробелы.
    groups = original.str.lower().str.replace(r'\s+', ' ', regex=True).str.strip().to_numpy()
    train_idx, valid_idx = make_baseline_split(target, groups, random_state=random_state)
    assert not set(groups[train_idx]).intersection(groups[valid_idx])
    assert set(target[valid_idx]).issubset(set(target[train_idx]))

    started = perf_counter()
    normalized = original.map(normalize_fn)
    if not normalized.map(lambda value: isinstance(value, str)).all():
        raise TypeError('Функция нормализации должна возвращать строку для каждой строки.')
    print(f'Нормализация: {(normalized != original).sum():,} изменённых строк; '
          f'{perf_counter() - started:.1f} с.')
    print(f'Train: {len(train_idx):,}; validation: {len(valid_idx):,}; '
          f'классов в train/validation: {len(np.unique(target[train_idx]))}/'
          f'{len(np.unique(target[valid_idx]))}. Seed = {random_state}.')
    print('Только StudentExplanation; цель — Category:Misconception. '
          'Классы, не попавшие в validation, этим сравнением не оцениваются.')

    results = []
    predictions = pd.DataFrame({
        'row_id': data['row_id'].to_numpy()[valid_idx] if 'row_id' in data else valid_idx,
        'target': target[valid_idx],
        'original': original.iloc[valid_idx].to_numpy(),
        'normalized': normalized.iloc[valid_idx].to_numpy(),
    })
    for name, texts in [('original', original), ('normalized', normalized)]:
        print(f'Обучение {name}...', flush=True)
        started = perf_counter()
        model = make_text_baseline(random_state=random_state)

        model.fit(texts.iloc[train_idx], target[train_idx])
        scores = model.decision_function(texts.iloc[valid_idx])
        classes = model.named_steps['classifier'].classes_
        if scores.shape != (len(valid_idx), len(classes)) or len(classes) < 3:
            raise ValueError('Классификатор должен вернуть оценки минимум для трёх классов.')
        top3 = classes[np.argsort(-scores, axis=1, kind='stable')[:, :3]]
        score = map_at_3(target[valid_idx], top3)
        results.append({
            'variant': name, 'MAP@3': score,
            'features': model.named_steps['classifier'].n_features_in_,
            'fit_predict_seconds': perf_counter() - started,
        })
        predictions[f'{name}_top3'] = list(map(tuple, top3))
        print(f'{name}: MAP@3 = {score:.6f}; {perf_counter() - started:.1f} с.', flush=True)

    results = pd.DataFrame(results).set_index('variant')
    print('\n' + results.to_string(float_format=lambda value: f'{value:.6f}'))
    delta = results.loc['normalized', 'MAP@3'] - results.loc['original', 'MAP@3']
    print(f'\nΔ MAP@3 (normalized − original): {delta:+.6f} ({delta * 100:+.3f} п.п.).')
    return results, predictions


if "normalize_numbers" not in globals():
    raise RuntimeError("Сначала выполните ячейку с функцией normalize_numbers.")

baseline_results, baseline_predictions = compare_normalization_baseline(normalize_numbers)


Нормализация: 13,462 изменённых строк; 3.1 с.
Train: 29,415; validation: 7,281; классов в train/validation: 65/51. Seed = 42.
Только StudentExplanation; цель — Category:Misconception. Классы, не попавшие в validation, этим сравнением не оцениваются.
Обучение original...
original: MAP@3 = 0.832097; 10.5 с.
Обучение normalized...
normalized: MAP@3 = 0.832326; 10.6 с.

              MAP@3  features  fit_predict_seconds
variant                                           
original   0.832097     82998            10.452710
normalized 0.832326     81162            10.634436

Δ MAP@3 (normalized − original): +0.000229 (+0.023 п.п.).
